# Multiclass support for sklearn classification tests

This sample exercises four ValidMind sklearn model-validation tests on both a **binary**
and a **multiclass** classification model:

- `ROCCurve`
- `PrecisionRecallCurve`
- `ConfusionMatrix`
- `PopulationStabilityIndex`

Each test keeps its original behavior for binary targets and gains explicit multiclass
handling:

- **ROCCurve / PrecisionRecallCurve** — render **one-vs-rest** curves for multiclass
  targets (one curve per class plus a micro-average), using the model's per-class
  `predict_proba`.
- **ConfusionMatrix** — renders the full N×N matrix. The `threshold` parameter only
  applies to binary targets; for multiclass the model's argmax class predictions are used.
- **PopulationStabilityIndex** — computes **one-vs-rest** PSI (one table/plot per class)
  from the model's per-class `predict_proba`.

Where a multiclass model can't produce a full per-class probability matrix (e.g.
metadata-only / Foundation models, or predictions supplied as a single precomputed
probability column), the ROC/PR/PSI tests **skip gracefully** rather than crashing.

> Runs fully offline — no ValidMind platform connection required (`generate_description=False`).

In [ ]:
# This notebook exercises UNRELEASED multiclass changes on this branch, so install
# THIS local checkout (editable) into the kernel — not the PyPI release:
%pip install -q -e ../..

# To test the published release instead (note: no multiclass ROC/PR/PSI support yet), use:
# %pip install -q validmind

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

import validmind as vm
from validmind.tests import run_test

ROC = "validmind.model_validation.sklearn.ROCCurve"
PR = "validmind.model_validation.sklearn.PrecisionRecallCurve"
CM = "validmind.model_validation.sklearn.ConfusionMatrix"
PSI = "validmind.model_validation.sklearn.PopulationStabilityIndex"


def build_inputs(n_classes, prefix):
    """Fit a RandomForest and return (model, train_dataset, test_dataset).

    PopulationStabilityIndex compares two datasets, so we keep the train split
    around too; ROC/PR/ConfusionMatrix only use the test dataset.
    """
    X, y = make_classification(
        n_samples=1500,
        n_features=8,
        n_informative=5,
        n_classes=n_classes,
        n_clusters_per_class=1,
        random_state=42,
    )
    cols = [f"f{i}" for i in range(X.shape[1])]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    train_df = pd.DataFrame(X_train, columns=cols)
    train_df["target"] = y_train
    test_df = pd.DataFrame(X_test, columns=cols)
    test_df["target"] = y_test

    train_ds = vm.init_dataset(
        input_id=f"{prefix}_train", dataset=train_df, target_column="target", __log=False
    )
    test_ds = vm.init_dataset(
        input_id=f"{prefix}_test", dataset=test_df, target_column="target", __log=False
    )

    clf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
    model = vm.init_model(input_id=f"{prefix}_model", model=clf, __log=False)
    train_ds.assign_predictions(model)
    test_ds.assign_predictions(model)
    return model, train_ds, test_ds

## 1. ROC & Precision-Recall curves

### 1a. Binary classification

Expect a single ROC curve (with AUC) and a single Precision-Recall curve — the original behavior.

In [ ]:
bin_model, bin_train_ds, bin_ds = build_inputs(n_classes=2, prefix="binary")

run_test(ROC, inputs={"model": bin_model, "dataset": bin_ds}, generate_description=False)

In [ ]:
run_test(PR, inputs={"model": bin_model, "dataset": bin_ds}, generate_description=False)

### 1b. Multiclass classification (one-vs-rest)

Expect **one curve per class plus a micro-average**. `ROCCurve` also draws the random
baseline; `PrecisionRecallCurve` reports average precision (AP) per class.

In [ ]:
mc_model, mc_train_ds, mc_ds = build_inputs(n_classes=4, prefix="multiclass")

run_test(ROC, inputs={"model": mc_model, "dataset": mc_ds}, generate_description=False)

In [ ]:
run_test(PR, inputs={"model": mc_model, "dataset": mc_ds}, generate_description=False)

## 2. Confusion matrix

The confusion matrix renders for any number of classes. For **binary** targets the cells
are labeled TN / FP / FN / TP and the `threshold` parameter splits the positive-class
probability. For **multiclass** targets it renders the full N×N matrix from the model's
argmax class predictions (`threshold` is ignored).

In [ ]:
run_test(CM, inputs={"model": bin_model, "dataset": bin_ds}, generate_description=False)

In [ ]:
run_test(CM, inputs={"model": mc_model, "dataset": mc_ds}, generate_description=False)

## 3. Population Stability Index

PSI compares the model's predicted-probability distribution between two datasets (here
train vs test). For **binary** targets it produces a single PSI table/plot on the
positive-class probability. For **multiclass** targets it produces **one-vs-rest** PSI —
one table and one subplot per class — from the model's per-class `predict_proba`.

In [ ]:
run_test(PSI, inputs={"model": bin_model, "datasets": [bin_train_ds, bin_ds]}, generate_description=False)

In [ ]:
run_test(PSI, inputs={"model": mc_model, "datasets": [mc_train_ds, mc_ds]}, generate_description=False)

## 4. Inspect the raw values

For multiclass, `RawData` is keyed per class (by label). ROC/PR also include a `micro`
entry. This confirms the one-vs-rest breakdown behind the plots.

In [ ]:
from validmind.tests.model_validation.sklearn.ROCCurve import ROCCurve
from validmind.tests.model_validation.sklearn.PrecisionRecallCurve import (
    PrecisionRecallCurve,
)
from validmind.tests.model_validation.sklearn.PopulationStabilityIndex import (
    PopulationStabilityIndex,
)

_, roc_raw = ROCCurve(mc_model, mc_ds)
print("ROC per-class + micro AUC:")
print({k: round(float(v), 3) for k, v in roc_raw.auc.items()})

_, pr_raw = PrecisionRecallCurve(mc_model, mc_ds)
print("\nPR per-class + micro average precision:")
print({k: round(float(v), 3) for k, v in pr_raw.average_precision.items()})

_, _, psi_raw = PopulationStabilityIndex([mc_train_ds, mc_ds], mc_model)
print("\nPSI one-vs-rest total per class:")
print({cls: round(rows[-1]["psi"], 4) for cls, rows in psi_raw.psi_raw.items()})

## 5. Graceful skip

When a multiclass model cannot provide a per-class probability matrix, the ROC/PR/PSI
tests raise `SkipTestError` instead of crashing. Below we simulate that with a model
whose `predict_proba` returns a single precomputed probability column.

In [ ]:
from validmind.errors import SkipTestError


# A model whose predict_proba can't yield a per-class matrix for these classes.
class SingleColumnProbaModel:
    def predict(self, X):
        return np.zeros(len(X), dtype=int)

    def predict_proba(self, X):
        return np.full(len(X), 0.5)  # 1-D: not a per-class matrix


skip_model = vm.init_model(
    input_id="skip_model", model=SingleColumnProbaModel(), __log=False
)

# The multiclass paths read the underlying model's predict_proba directly, so no
# assign_predictions is needed here — the missing per-class matrix triggers the skip.
try:
    ROCCurve(skip_model, mc_ds)
except SkipTestError as e:
    print("ROCCurve skipped as expected:", e)

try:
    PopulationStabilityIndex([mc_train_ds, mc_ds], skip_model)
except SkipTestError as e:
    print("PopulationStabilityIndex skipped as expected:", e)

<!-- VALIDMIND COPYRIGHT -->

<small>

***

Copyright © 2023-2026 ValidMind Inc. All rights reserved.<br>
Refer to [LICENSE](https://github.com/validmind/validmind-library/blob/main/LICENSE) for details.<br>
SPDX-License-Identifier: AGPL-3.0 AND ValidMind Commercial</small>